In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# Extract the raw dataset

In [3]:
df = pd.read_csv("ecommerce_sales_analytics_5000.csv")

In [4]:
df.head()

,order_id,order_date,customer_id,product_category,region,quantity,unit_price,discount,payment_method,delivery_days,customer_rating,revenue
0,10001,1/1/2022,1102,Beauty,South,7,373.65,0.28,Wallet,10,4.7,1883.20
1,10002,1/2/2022,1435,Clothing,South,7,47.74,0.09,Card,6,3.9,304.10
2,10003,1/3/2022,1860,Beauty,East,3,311.28,0.31,COD,6,2.5,644.35
3,10004,1/4/2022,1270,Electronics,West,5,524.47,0.02,Wallet,6,1.6,2569.90
4,10005,1/5/2022,1106,Clothing,West,5,139.87,0.33,Wallet,4,4.9,468.56


In [5]:
df.shape

(5000, 12)

In [11]:
df.columns.tolist()

['order_id',
 'order_date',
 'customer_id',
 'product_category',
 'region',
 'quantity',
 'unit_price',
 'discount',
 'payment_method',
 'delivery_days',
 'customer_rating',
 'revenue']

# Inspect the data

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          5000 non-null   int64  
 1   order_date        5000 non-null   object 
 2   customer_id       5000 non-null   int64  
 3   product_category  5000 non-null   object 
 4   region            5000 non-null   object 
 5   quantity          5000 non-null   int64  
 6   unit_price        5000 non-null   float64
 7   discount          5000 non-null   float64
 8   payment_method    5000 non-null   object 
 9   delivery_days     5000 non-null   int64  
 10  customer_rating   5000 non-null   float64
 11  revenue           5000 non-null   float64
dtypes: float64(4), int64(4), object(4)
memory usage: 468.9+ KB


In [14]:
df.isnull().sum()

order_id            0
order_date          0
customer_id         0
product_category    0
region              0
quantity            0
unit_price          0
discount            0
payment_method      0
delivery_days       0
customer_rating     0
revenue             0
dtype: int64

In [15]:
df.duplicated().sum()

np.int64(0)

In [17]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
order_id,5000.0,12500.500000,1443.520003,10001.00,11250.7500,12500.50,13750.2500,15000.00
customer_id,5000.0,1505.701200,290.836902,1000.00,1253.0000,1510.00,1761.0000,1999.00
quantity,5000.0,4.044800,2.020398,1.00,2.0000,4.00,6.0000,7.00
unit_price,5000.0,308.418774,169.259369,15.15,161.8950,309.89,455.5575,599.96
discount,5000.0,0.179984,0.101404,0.00,0.0900,0.18,0.2700,0.35
delivery_days,5000.0,6.118800,3.153264,1.00,3.0000,6.00,9.0000,11.00
customer_rating,5000.0,2.973980,1.157722,1.00,2.0000,3.00,4.0000,5.00
revenue,5000.0,1021.955148,825.584219,11.21,354.5275,796.65,1515.6900,4119.33


In [21]:
quality_report = pd.DataFrame({
    "column" : df.columns,
    "datatype" : df.dtypes.astype(str).values,
    "missing_values" : df.isnull().sum().values,
    "unique_values" : df.nunique().values
})
quality_report

,column,datatype,missing_values,unique_values
0,order_id,int64,0,5000
1,order_date,object,0,5000
2,customer_id,int64,0,989
3,product_category,object,0,4
4,region,object,0,4
5,quantity,int64,0,7
6,unit_price,float64,0,4788
7,discount,float64,0,36
8,payment_method,object,0,3
9,delivery_days,int64,0,11


# Validate the data

In [22]:
df['order_id'].duplicated().sum()

np.int64(0)

In [23]:
print("Negative Quantities : " , (df["quantity"]<0).sum())
print("Negative unit price : " , (df["unit_price"]<0).sum())
print("Negative revenue : " , (df["revenue"]<0).sum())
print("Negative delivery days : " , (df["delivery_days"]<0).sum())

Negative Quantities :  0
Negative unit price :  0
Negative revenue :  0
Negative delivery days :  0


In [24]:
print("Invalid ratings : " , ((df["customer_rating"]<1) | (df["customer_rating"]>5)).sum())

Invalid ratings :  0


# Clean the data

In [25]:
df["order_date"] = pd.to_datetime(df["order_date"],errors="coerce")

# Create transformed columns

In [27]:
df["year"] = df["order_date"].dt.year

In [28]:
df["month"] = df["order_date"].dt.month

In [29]:
df["quarter"] = df["order_date"].dt.quarter

In [30]:
df["gross_sales"] = (df["quantity"]*df["unit_price"])

In [31]:
df["discount_amount"] = (df["gross_sales"] * df["discount"])

In [32]:
df["calculated_revenue"] = (df["gross_sales"] - df["discount_amount"])

# Validate revenue

In [33]:
df["revenue_difference"] = (df['revenue'] - df['calculated_revenue'])

In [36]:
df["revenue_difference"].describe()

count    5000.000000
mean       -0.000029
std         0.002831
min        -0.005000
25%        -0.002400
50%         0.000000
75%         0.002400
max         0.005000
Name: revenue_difference, dtype: float64

# Create a final dataset

In [37]:
final_df = df.drop(columns=["revenue_difference"])

In [38]:
final_df.head()

,order_id,order_date,customer_id,product_category,region,quantity,unit_price,discount,payment_method,delivery_days,customer_rating,revenue,year,month,quarter,gross_sales,discount_amount,calculated_revenue
0,10001,2022-01-01,1102,Beauty,South,7,373.65,0.28,Wallet,10,4.7,1883.20,2022,1,1,2615.55,732.3540,1883.1960
1,10002,2022-01-02,1435,Clothing,South,7,47.74,0.09,Card,6,3.9,304.10,2022,1,1,334.18,30.0762,304.1038
2,10003,2022-01-03,1860,Beauty,East,3,311.28,0.31,COD,6,2.5,644.35,2022,1,1,933.84,289.4904,644.3496
3,10004,2022-01-04,1270,Electronics,West,5,524.47,0.02,Wallet,6,1.6,2569.90,2022,1,1,2622.35,52.4470,2569.9030
4,10005,2022-01-05,1106,Clothing,West,5,139.87,0.33,Wallet,4,4.9,468.56,2022,1,1,699.35,230.7855,468.5645


In [39]:
final_df.shape

(5000, 18)

# Save processed data

In [40]:
final_df.to_csv("cleaned_data.csv",index=False)